In [0]:
# Databricks notebook source
# SILVER LAYER: Data Quality Validation
# These are stricter than Bronze — Silver should be near-perfect

from pyspark.sql.functions import col, count, when, isnan, lit, current_timestamp

fact = spark.read.table("f1_data.silver.fact_results")
drivers = spark.read.table("f1_data.silver.dim_drivers")
sessions = spark.read.table("f1_data.silver.dim_sessions")
meetings = spark.read.table("f1_data.silver.dim_meetings")

# ============ CHECK 1: REFERENTIAL INTEGRITY ============
# Every result should have a matched driver (no nulls from failed joins)
unmatched_drivers = fact.filter(col("driver_found_in_dim") == False).count()

print(f"\n{'='*60}")
print(f"REFERENTIAL INTEGRITY")
print(f"  Unmatched drivers in fact_results: {unmatched_drivers}")
print(f"  {'✓ PASS' if unmatched_drivers == 0 else '⚠️  FAIL'}")

# ============ CHECK 2: BUSINESS RULE VALIDATION ============
# DNF + DNS + DSQ can't ALL be true at once
impossible_status = fact.filter(
    (col("dnf") == True) & (col("dns") == True)
).count()

print(f"\nBUSINESS RULE VALIDATION")
print(f"  DNF + DNS simultaneously: {impossible_status}")
print(f"  {'✓ PASS' if impossible_status == 0 else '⚠️  FAIL'}")

# ============ CHECK 3: POINTS/POSITION CONSISTENCY ============
# Points should be 0 if driver DNF'd, DNS'd, or DSQ'd
# (unless fastest lap bonus point — adjust if needed)
inconsistent_points = fact.filter(
    ((col("dnf") == True) | (col("dns") == True) | (col("dsq") == True)) &
    (col("points") > 1)  # >1 because some series award fastest lap bonus point
).count()

print(f"\nPOINTS/STATUS CONSISTENCY")
print(f"  Records with DNF/DNS/DSQ and >1 point: {inconsistent_points}")
print(f"  {'✓ PASS' if inconsistent_points == 0 else '⚠️  REVIEW (may be fastest lap point)'}")

# ============ CHECK 4: DUPLICATE FACT RECORDS ============
total_facts = fact.count()
unique_facts = fact.dropDuplicates(["driver_number", "meeting_key", "session_key"]).count()

print(f"\nDUPLICATE CHECK")
print(f"  Total records: {total_facts}")
print(f"  Unique by composite key: {unique_facts}")
print(f"  Duplicates: {total_facts - unique_facts}")
print(f"  {'✓ PASS' if total_facts == unique_facts else '⚠️  FAIL'}")

# ============ CHECK 5: PARTITION COVERAGE ============
# count the number of occurrences (rows) for each unique combination of year and session type, sorts the results chronologically and 
# alphabetically, and displays them as a formatted table
print(f"\nPARTITION COVERAGE")
fact.groupBy("year", "session_type").count().orderBy("year", "session_type").show()

# ============ CHECK 6: SILVER TABLE COMPLETENESS ============
print(f"\nTABLE COMPLETENESS")
print(f"  dim_drivers:  {drivers.count()} records")
print(f"  dim_sessions: {sessions.count()} records")
print(f"  dim_meetings: {meetings.count()} records")
print(f"  fact_results: {fact.count()} records")

# ============ LOG TO SILVER DQ TABLE ============
dq_results = [
    ("fact_results", "referential_integrity", "driver_found_in_dim", unmatched_drivers, 0, unmatched_drivers == 0),
    ("fact_results", "business_rule", "dnf_dns_simultaneous", impossible_status, 0, impossible_status == 0),
    ("fact_results", "points_consistency", "points_with_dnf", inconsistent_points, 0, inconsistent_points == 0),
    ("fact_results", "duplicate_check", "composite_key", total_facts - unique_facts, 0, total_facts == unique_facts),
]

dq_df = spark.createDataFrame(
    dq_results,
    ["table_name", "check_type", "column", "value", "threshold", "pass"]
).withColumn("timestamp", current_timestamp())

dq_df.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable(
    "f1_data.silver._data_quality_log"
)

failures = dq_df.filter(col("pass") == False)
if failures.count() > 0:
    print(f"\n⚠️  {failures.count()} DQ checks FAILED in Silver layer")
    failures.display()
    raise Exception("Silver DQ checks failed — investigate before Gold layer")
else:
    print(f"\n✅ All Silver DQ checks PASSED")


REFERENTIAL INTEGRITY
  Unmatched drivers in fact_results: 0
  ✓ PASS

BUSINESS RULE VALIDATION
  DNF + DNS simultaneously: 0
  ✓ PASS

POINTS/STATUS CONSISTENCY
  Records with DNF/DNS/DSQ and >1 point: 0
  ✓ PASS

DUPLICATE CHECK
  Total records: 419
  Unique by composite key: 419
  Duplicates: 0
  ✓ PASS

PARTITION COVERAGE
+----+------------+-----+
|year|session_type|count|
+----+------------+-----+
|2024|        Race|  419|
+----+------------+-----+


TABLE COMPLETENESS
  dim_drivers:  9936 records
  dim_sessions: 123 records
  dim_meetings: 25 records
  fact_results: 419 records

✅ All Silver DQ checks PASSED
